# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [5]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), feature_columns),
    ]
)

scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            feature_columns,
        ),
    ]
)

**Respuesta:** El `StandardScaler` debe estar dentro del pipeline porque debe ajustarse únicamente con los datos de entrenamiento de cada fold. Si se ajustara antes de la validación cruzada, usaría información del conjunto de validación y produciría fuga de datos, haciendo que la métrica fuese demasiado optimista.

## 3. Modelos candidatos

In [6]:
models = {
    "decision_tree": Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]
    ),
    "logistic_regression": Pipeline(
        steps=[
            ("preprocessor", scaled_preprocessor),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    ),
    "knn": Pipeline(
        steps=[
            ("preprocessor", scaled_preprocessor),
            ("model", KNeighborsClassifier()),
        ]
    ),
}

## 4. Comparación inicial con validación cruzada

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

comparison_rows = []
for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    comparison_rows.append(
        {
            "model": model_name,
            "train_accuracy_mean": scores["train_score"].mean(),
            "validation_accuracy_mean": scores["test_score"].mean(),
            "validation_accuracy_std": scores["test_score"].std(),
        }
    )

initial_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("validation_accuracy_mean", ascending=False)
    .reset_index(drop=True)
)
initial_comparison

,model,train_accuracy_mean,validation_accuracy_mean,validation_accuracy_std
0,logistic_regression,1.0000,0.979064,0.027713
1,knn,0.9771,0.950739,0.035274
2,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [8]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [None, 3, 5, 7],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {
        "model__C": [0.1, 1.0, 10.0],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
tuned_rows = []
for model_name, model in models.items():
    search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    searches[model_name] = search
    tuned_rows.append(
        {
            "model": model_name,
            "best_score": search.best_score_,
            "best_params": search.best_params_,
        }
    )

tuned_comparison = (
    pd.DataFrame(tuned_rows)
    .sort_values("best_score", ascending=False)
    .reset_index(drop=True)
)
tuned_comparison

,model,best_score,best_params
0,logistic_regression,0.979064,{'model__C': 1.0}
1,knn,0.957882,"{'model__n_neighbors': 7, 'model__weights': 'u..."
2,decision_tree,0.902463,"{'model__max_depth': None, 'model__min_samples..."


## 6. Selección y persistencia del campeón

In [9]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion_name": champion_name,
    "best_score": searches[champion_name].best_score_,
    "best_params": searches[champion_name].best_params_,
    "random_state": RANDOM_STATE,
    "cv_folds": cv.n_splits,
    "scoring": "accuracy",
    "features": feature_columns,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2),
    encoding="utf-8",
)

cv_results = []
for model_name, search in searches.items():
    model_results = pd.DataFrame(search.cv_results_)
    model_results.insert(0, "model", model_name)
    cv_results.append(model_results)

pd.concat(cv_results, ignore_index=True).to_csv(
    REPORTS_DIR / "cv_results.csv",
    index=False,
)

print("Campeón:", champion_name)
print("Accuracy CV:", searches[champion_name].best_score_)

Campeón: logistic_regression
Accuracy CV: 0.9790640394088669


In [10]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
-  x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.